# DEMO-1 retrieval, citation và Chroma

Chưa chạy. Run All trên Colab với source + input của bước trước đã lưu bền vững. Không dùng dữ liệu trường/private. Đọc ml/COLAB_RUNBOOK.md.


In [ ]:
from pathlib import Path
import sys, subprocess, zipfile
REPO = Path("/content/student-advisor")
# Upload ONLY the source ZIP prepared for this project, never .env or private student data.
if not (REPO / "ml").exists():
    from google.colab import files
    uploaded = files.upload()
    source_zip = next((Path(name) for name in uploaded if name.endswith(".zip")), None)
    assert source_zip, "Upload student-advisor-colab-source.zip"
    with zipfile.ZipFile(source_zip) as archive:
        for member in archive.infolist():
            target = (REPO / member.filename).resolve()
            assert target.is_relative_to(REPO.resolve()), "Unsafe ZIP path"
            assert not (member.external_attr >> 16 & 0o170000) == 0o120000, "Symlink not allowed"
        archive.extractall(REPO)
assert (3, 12) <= sys.version_info[:2] < (3, 14), f"Use Python 3.12 or 3.13; current={sys.version.split()[0]}. Align/export environment with backend before final packaging."
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO/"ml/requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", str(REPO/"packages/advisor_core")], check=True)
sys.path.insert(0, str(REPO))
# Optional: set True yourself and select your own project folder. No automatic Drive upload.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
WORK = Path("/content/drive/MyDrive/student-advisor-work") if USE_DRIVE else Path("/content/work")
WORK.mkdir(parents=True, exist_ok=True)
print("WORK:", WORK, "Ephemeral runtime: download outputs before disconnecting." if not USE_DRIVE else "")
from ml.scripts.common import load, save, sha
from ml.scripts.bootstrap import record_environment
record_environment(REPO, WORK)
config = load(REPO/"ml/configs/experiment.json")


In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-r",str(REPO/"ml/requirements-rag.txt")],check=True)
from ml.scripts.rag_eval import prepare_corpus, evaluate, write_final_approval_template
from advisor_core.rag import lexical_search, evidence_response
rag_config=load(REPO/"ml/rag/config.json")
chunks=prepare_corpus(REPO/"ml/rag/corpus/DEMO1.md",WORK/"rag",rag_config)
print(evidence_response(lexical_search(chunks,"Học lại tính điểm như thế nào?","demo_academic","2026-09-06"),"Học lại"))
from shutil import copyfile
BENCHMARK_DRAFT=REPO/"ml/rag/questions_benchmark_draft.jsonl"
REVIEWED_QUESTIONS=WORK/"rag/questions_reviewed.jsonl"
if not REVIEWED_QUESTIONS.exists():
    copyfile(BENCHMARK_DRAFT, REVIEWED_QUESTIONS)
print("Review all 120 rows in", REVIEWED_QUESTIONS, "before setting any reviewed=true.")
from huggingface_hub import HfApi
from advisor_core.rag import DenseRetriever
rag_config["embedding_revision"]=HfApi().model_info(rag_config["embedding_model"]).sha
save(WORK/"rag/rag_config_pinned.json",rag_config)
dense= DenseRetriever(WORK/"rag/chroma",rag_config)
print(dense.index(chunks))
print(evidence_response(dense.search("Học lại tính điểm như thế nào?","demo_academic","2026-09-06"),"Học lại"))
RUN_REVIEWED_BENCHMARK=False  # set True only after every row has been owner-reviewed
if RUN_REVIEWED_BENCHMARK:
    from huggingface_hub import HfApi
    # Pin the exact model revision before loading; no remote custom code.
    assert rag_config["embedding_revision"], "Use the revision already pinned above"
    save(WORK/"rag/rag_config_pinned.json",rag_config)
    print(evaluate(chunks,REVIEWED_QUESTIONS,rag_config,WORK/"rag/dev-run-001",
                   split="dev",dense=True,index_path=WORK/"rag/chroma",reviewed=True))
PREPARE_FINAL_RAG_APPROVAL=False
if PREPARE_FINAL_RAG_APPROVAL:
    approval_template=WORK/"rag/rag_final_approval.template.json"
    write_final_approval_template(REVIEWED_QUESTIONS,rag_config,approval_template)
    print(f"Review/copy {approval_template} to rag_final_approval.json, set approved=true, then set RUN_FINAL_RAG_BENCHMARK=True in a later execution.")
RUN_FINAL_RAG_BENCHMARK=False  # must remain False until dev results and approval are accepted
if RUN_FINAL_RAG_BENCHMARK:
    print(evaluate(chunks,REVIEWED_QUESTIONS,rag_config,WORK/"rag/test-run-001",
                   split="test",dense=True,index_path=WORK/"rag/chroma",reviewed=True,
                   freeze_approval_path=WORK/"rag/rag_final_approval.json"))
print("Generation API disabled. Review rubric/gold and keep final test separate.")
